# 1. Package Imports

In [0]:
import re
import logging
import pyspark.sql.functions as F

# 2. Dataset Config

In [0]:

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("Silver_Layer_Dam_Levels")

ds_config ={
    "bronze_table": "cpt_utility_catalog.bronze.bronze_dam_levels_raw",
    "silver_table": "cpt_utility_catalog.silver.silver_dam_levels_cleaned",
    "changes":{
        "headers":{
            "drop_column": ["ObjectId"],
        },
        "columns":{
            "trim_whitespace": True,
            "fill_na_value": None,
            "drop_duplicates": True,
        },
    }
}

df_new = spark.table(ds_config["bronze_table"])
hdr_config = ds_config["changes"]["headers"]
col_config = ds_config["changes"]["columns"]

logger.info("Silver layer Dam levels table configuration loaded")


# 3. Dataset Cleaning

## 3.1 Drop Columns

In [0]:
drop_list = hdr_config["drop_column"]
if isinstance(drop_list, list):
    df_new = df_new.drop(*drop_list)

## 3.2 Clean Column Names

In [0]:
# renaming headers in [name]_[metric]_[unit] unit format
renamed_headers = [col.rstrip("_").lower().replace("last","_last")
                .replace("ë","e").replace("ml","Ml").replace("current", "current_pct")
                .replace("year","year_pct").replace("___","_")
                .replace("__","_") for col in df_new.columns]
                
# creating new dict() old_header_name : new_header_name
new_header_map = {old_header : new_header for old_header, new_header in zip(df_new.columns, renamed_headers)}

# renaming all headers at once
df_new = df_new.withColumnsRenamed(new_header_map)